# Day 4 · Exercise 5: Retry Loop

**What you'll build:** `extract_with_retry` — add self-correction: when validation fails, feed the error back to the model and try again.

**Why it matters:** Even with all three layers in place, validation fails ~5–10% of the time. A retry loop — with the error fed back as context — fixes almost all remaining failures. This is the difference between a prototype and a production-ready pipeline.

## Setup (already defined)

In [ ]:
import ollama
import json
from pydantic import BaseModel, Field, ValidationError

MODEL = "llama3.2"

class PersonProfile(BaseModel):
    name:   str       = Field(description="Full name of the person")
    age:    int       = Field(description="Age in years")
    city:   str       = Field(description="City where they live")
    skills: list[str] = Field(description="List of professional skills")
    bio:    str       = Field(description="One-sentence biography")

_schema_str = json.dumps(PersonProfile.model_json_schema(), indent=2)
_example_str = (
    '{"name": "Jane Smith", "age": 28, "city": "Cape Town", '
    '"skills": ["Python", "SQL"], '
    '"bio": "A data engineer who builds data pipelines."}'
)
SYSTEM = (
    "You are an information extractor. Read the user's text and extract "
    "the information into a JSON object with exactly these fields:\n\n"
    f"{_schema_str}\n\n"
    "Fill each field with the ACTUAL VALUE from the text — not the schema.\n"
    f"Example of a correctly filled response:\n{_example_str}\n\n"
    "Return only the JSON object — no explanation, no prose."
)

## Your Implementation

In [ ]:
def extract_with_retry(text: str, max_tries: int = 3) -> PersonProfile:
    """Extract a PersonProfile with automatic retry on validation failure.

    On each attempt:
    - Call ollama.chat with format="json" and the schema system prompt
    - Try PersonProfile.model_validate_json() on the response
    - If it succeeds, return the profile immediately
    - If ValidationError is raised and retries remain:
        * Append the model's bad response as an assistant turn
        * Append a user turn explaining the error and asking for a fix
        * Continue to the next attempt
    - If all retries are exhausted, re-raise the final ValidationError

    Args:
        text:      A paragraph describing a person.
        max_tries: Maximum number of attempts (default 3).

    Returns:
        A validated PersonProfile instance.

    Raises:
        ValidationError: if all retries are exhausted.

    Example:
        profile = extract_with_retry(
            "Bob Mokoena, 28, ML engineer in Johannesburg. Skills: PyTorch, FastAPI.",
            max_tries=3,
        )
    """
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": text},
    ]
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _ollama_running():
    try:
        import urllib.request  # stdlib — no install needed
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

TEXT = (
    "Bob Mokoena is a 28-year-old machine learning engineer based in Johannesburg. "
    "He works with PyTorch, Scikit-learn, and FastAPI. "
    "Bob builds recommendation systems for e-commerce platforms."
)

def _run_checks():
    score, total = 0, 5

    # Check 1: function exists and accepts max_tries
    try:
        assert callable(extract_with_retry), 'extract_with_retry is not defined'
        import inspect
        sig = inspect.signature(extract_with_retry)
        assert 'max_tries' in sig.parameters, 'function should accept max_tries parameter'
        print(f'{_PASS} Check 1/{total}: function exists and accepts max_tries parameter')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: Ollama is running
    if not _ollama_running():
        print(f'{_FAIL} Check 2/{total}: Ollama server is not running')
        print('  → macOS: open the Ollama app · Linux/Windows: run ollama serve')
        return
    print(f'{_PASS} Check 2/{total}: Ollama server is reachable')
    score += 1

    # Check 3: returns a PersonProfile
    profile = None
    try:
        profile = extract_with_retry(TEXT, max_tries=3)
        assert isinstance(profile, PersonProfile), \
            f'expected PersonProfile, got {type(profile).__name__}'
        print(f'{_PASS} Check 3/{total}: returned a PersonProfile instance')
        score += 1
    except ValidationError as e:
        print(f'{_FAIL} Check 3/{total}: ValidationError after all retries — {e}')
    except AssertionError as e:
        print(f'{_FAIL} Check 3/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: call failed — {e}')

    if profile is None:
        print(f'  {score}/{total} passed. Keep going!')
        return

    # Check 4: name is a str
    try:
        assert isinstance(profile.name, str) and len(profile.name) > 0
        print(f'{_PASS} Check 4/{total}: profile.name is a non-empty str ({profile.name!r})')
        score += 1
    except AssertionError:
        print(f'{_FAIL} Check 4/{total}: profile.name is {profile.name!r}')

    # Check 5: age is an int
    try:
        assert isinstance(profile.age, int) and profile.age > 0
        print(f'{_PASS} Check 5/{total}: profile.age is a positive int ({profile.age})')
        score += 1
    except AssertionError:
        print(f'{_FAIL} Check 5/{total}: profile.age is {profile.age!r} (type: {type(profile.age).__name__})')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 5 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Test that retries actually work by temporarily patching the first call to return bad JSON:

```python
# Simulate a first-attempt failure by passing deliberately ambiguous text
# and see if the retry self-corrects:
ambiguous = "Her name might be Sarah or Sara, she's somewhere between 25 and 30, "\
             "works in data somehow, lives in a big city."

try:
    profile = extract_with_retry(ambiguous, max_tries=3)
    print(f"Got: {profile}")
except ValidationError as e:
    print(f"All retries failed: {e}")
```

Then add a `print(f"Attempt {attempt+1}")` inside your loop to see how many attempts were needed.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def extract_with_retry(text: str, max_tries: int = 3) -> PersonProfile:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": text},
    ]
    for attempt in range(max_tries):
        response = ollama.chat(model=MODEL, messages=messages, format="json")
        raw = response["message"]["content"]
        try:
            return PersonProfile.model_validate_json(raw)
        except ValidationError as e:
            if attempt == max_tries - 1:
                raise  # all retries exhausted
            messages.append({"role": "assistant", "content": raw})
            messages.append({
                "role": "user",
                "content": (
                    f"That response failed validation:\n{e}\n\n"
                    "Please fix it and return only the corrected JSON object."
                ),
            })
    raise RuntimeError("unreachable")
```

**Key details:** (1) We re-raise on `attempt == max_tries - 1` so the final error propagates cleanly. (2) We append TWO messages on failure — the assistant's bad response and then a user correction — to keep the conversation structure valid (assistant and user must alternate). (3) The error message from `ValidationError` is included verbatim — the model reads it and corrects the specific problem. (4) `raise RuntimeError("unreachable")` is there only for type checkers — the loop always either returns or raises before reaching it.
</details>